# Missing Values  Imputation Function using ML

## Steps:
1. import the libraries
2. load the dataset
3. find thecolumns with missing values and store in an object
4. find the colums based on data type
    1. numeric
    2. categorical
    3. Boolean
 5. define the function to impute the mising values 
 6. apply the function to our dataset with missing values
 7. Check the missing values after imputation

In [ ]:
# import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import mean_absolute_error, accuracy_score, precision_score
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.model_selection import train_test_split

In [ ]:
# data loading
df = pd.read_csv('heart_disease_uci.csv')
df.head()

,id,age,sex,dataset,cp,trestbps,chol,fbs,restecg,thalch,exang,oldpeak,slope,ca,thal,num
0,1,63,Male,Cleveland,typical angina,145.0,233.0,True,lv hypertrophy,150.0,False,2.3,downsloping,0.0,fixed defect,0
1,2,67,Male,Cleveland,asymptomatic,160.0,286.0,False,lv hypertrophy,108.0,True,1.5,flat,3.0,normal,2
2,3,67,Male,Cleveland,asymptomatic,120.0,229.0,False,lv hypertrophy,129.0,True,2.6,flat,2.0,reversable defect,1
3,4,37,Male,Cleveland,non-anginal,130.0,250.0,False,normal,187.0,False,3.5,downsloping,0.0,normal,0
4,5,41,Female,Cleveland,atypical angina,130.0,204.0,False,lv hypertrophy,172.0,False,1.4,upsloping,0.0,normal,0


In [ ]:
# display list of those column that have missing values
df.isnull().sum()[df.isnull().sum()>0].sort_values(ascending=False)
missing_data_cols = df.isnull().sum()[df.isnull().sum()>0].index.tolist()
missing_data_cols


['trestbps',
 'chol',
 'fbs',
 'restecg',
 'thalch',
 'exang',
 'oldpeak',
 'slope',
 'ca',
 'thal']

In [ ]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 920 entries, 0 to 919
Data columns (total 16 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   id        920 non-null    int64  
 1   age       920 non-null    int64  
 2   sex       920 non-null    str    
 3   dataset   920 non-null    str    
 4   cp        920 non-null    str    
 5   trestbps  861 non-null    float64
 6   chol      890 non-null    float64
 7   fbs       830 non-null    object 
 8   restecg   918 non-null    str    
 9   thalch    865 non-null    float64
 10  exang     865 non-null    object 
 11  oldpeak   858 non-null    float64
 12  slope     611 non-null    str    
 13  ca        309 non-null    float64
 14  thal      434 non-null    str    
 15  num       920 non-null    int64  
dtypes: float64(5), int64(3), object(2), str(6)
memory usage: 115.1+ KB


In [ ]:
# find only categorical columns 
cat_cols = df.select_dtypes(include='object').columns.tolist()

# find only the numeric columns
num_cols = df.select_dtypes(exclude='object').columns.tolist()

print(f"Categorical Columns: {cat_cols}")
print(f"Numerical Columns: {num_cols}")

Categorical Columns: ['sex', 'dataset', 'cp', 'fbs', 'restecg', 'exang', 'slope', 'thal']
Numerical Columns: ['id', 'age', 'trestbps', 'chol', 'thalch', 'oldpeak', 'ca', 'num']


C:\Users\Zone Tech\AppData\Local\Temp\ipykernel_6236\2051195731.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = df.select_dtypes(include='object').columns.tolist()


In [ ]:
# categorical_cols = ['thal', 'ca', 'slope', 'exang', 'restecg', 'fbs', 'cp', 'sex', 'num']
# bool_cols = ['fbs',  'exang']
# numeric_cols = ['oldpeak', 'thalch', 'chol', 'trestbps', 'age']

In [ ]:
# define the function to impute the missing vaalues 
def impute_categorical_missing_data(df, target_column):
    """
    Fill missing values in any column using Machine Learning.

    Parameters:
        df : DataFrame
        target_column : Column whose missing values are to be filled

    Returns:
        DataFrame with missing values imputed
    """

    if df is None:
        raise ValueError("DataFrame is None. Please pass a valid DataFrame.")

    df = df.copy()

    # Encode object columns
    encoders = {}

    for col in df.select_dtypes(include=['object', 'category']).columns:
        le = LabelEncoder()
        df[col] = df[col].astype(str)
        df[col] = le.fit_transform(df[col])
        encoders[col] = le

    # Separate known and missing rows
    train_data = df[df[target_column].notnull()]
    test_data = df[df[target_column].isnull()]

    # If no missing values
    if len(test_data) == 0:
        print(f"No missing values found in '{target_column}'")
        return df

    # Features
    X_train = train_data.drop(columns=[target_column])
    y_train = train_data[target_column]

    X_test = test_data.drop(columns=[target_column])

    # Select model
    if y_train.dtype == 'object' or y_train.nunique() < 15:
        model = RandomForestClassifier(random_state=42)
    else:
        model = RandomForestRegressor(random_state=42)

    # Train
    model.fit(X_train, y_train)

    # Predict missing values
    predicted_values = model.predict(X_test)

    # Fill missing values
    df.loc[df[target_column].isnull(), target_column] = predicted_values

    return df

In [ ]:
# Impute missing values for each column

for col in missing_data_cols:
    df = impute_categorical_missing_data(df, col)

C:\Users\Zone Tech\AppData\Local\Temp\ipykernel_6236\1576020218.py:22: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in df.select_dtypes(include=['object', 'category']).columns:


No missing values found in 'fbs'
No missing values found in 'restecg'
No missing values found in 'exang'
No missing values found in 'slope'
No missing values found in 'thal'


In [ ]:
# let's impute the other columns with missing values 
(df.isnull().sum()/ len(df) * 100).sort_values(ascending=False)


id          0.0
age         0.0
sex         0.0
dataset     0.0
cp          0.0
trestbps    0.0
chol        0.0
fbs         0.0
restecg     0.0
thalch      0.0
exang       0.0
oldpeak     0.0
slope       0.0
ca          0.0
thal        0.0
num         0.0
dtype: float64